# 1. Phase 3 objective and gate philosophy

Learn recurrent 2v2/3v2 team play, then improve cross-physics robustness without losing nominal competence. Gate A blocks training when the benchmark is permissive; Gate B blocks robustness work when nominal cooperation is absent; Gate C blocks final seeds when competence-constrained fine-tuning fails. Smoke results never authorize scientific training.

In [ ]:
print('Phase 3 is staged, gated, and report-first. No cell silently launches training.')

# 2. Runtime variables

In [ ]:
from pathlib import Path
import json, os, shutil, sys
REPOSITORY_URL = os.environ.get('ROBOSOCCER_REPOSITORY_URL', '')
REPO_DIR = Path('/content/robot-soccer-transfer')
DRIVE_PROJECT = Path('/content/drive/MyDrive/RobotSoccerTransfer')
RUNS_ROOT = REPO_DIR / 'runs'
DEV_SEED = 3
NUM_ENVS = 16
CALIBRATION_EPISODES = 100
FINAL_CONFIRMATION_ENABLED = False

# 3. L4 and CUDA inspection

In [ ]:
!nvidia-smi
import torch
print({'torch': torch.__version__, 'cuda': torch.cuda.is_available(), 'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})

# 4. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)

# 5. Clone or fast-forward pull

In [ ]:
if (REPO_DIR / '.git').is_dir():
    %cd /content/robot-soccer-transfer
    !git pull --ff-only
else:
    assert REPOSITORY_URL, 'Set ROBOSOCCER_REPOSITORY_URL before cloning.'
    !git clone "$REPOSITORY_URL" /content/robot-soccer-transfer
%cd /content/robot-soccer-transfer

# 6. Install package

In [ ]:
!python -u -m pip install -e '.[dev]' --no-deps
!python -u -m pip install pymunk==7.3.0 pettingzoo==1.26.1 gymnasium==1.2.1 tqdm==4.67.1

# 7. Restore selected Phase 2 artifacts

In [ ]:
from robosoccer.artifacts import merge_artifact_directory, sync_artifacts_from_drive
sync_artifacts_from_drive(DRIVE_PROJECT, REPO_DIR, include_training_artifacts=False)
phase2_selected = ['20260722_002437_mappo_nominal_mappo_seed0', '20260722_020456_mappo_uniform_dr_mappo_seed0', '20260722_033215_mappo_failure_dr_mappo_seed0']
for run_name in phase2_selected:
    source = DRIVE_PROJECT / 'runs' / run_name
    if source.is_dir():
        merge_artifact_directory(source, RUNS_ROOT / run_name, include_training_artifacts=True)
for source in sorted((DRIVE_PROJECT / 'runs').glob('*phase3*')):
    metadata_path = source / 'run_metadata.json'
    if metadata_path.is_file() and json.loads(metadata_path.read_text()).get('status') == 'complete':
        merge_artifact_directory(source, RUNS_ROOT / source.name, include_training_artifacts=True)
def phase3_stage_run(stage):
    matches = []
    for run_dir in RUNS_ROOT.iterdir():
        metadata_path = run_dir / 'run_metadata.json'
        if not metadata_path.is_file():
            continue
        metadata = json.loads(metadata_path.read_text())
        active = metadata.get('resolved_configuration', {}).get('phase3', {}).get('active_stage')
        if metadata.get('status') == 'complete' and metadata.get('experiment_name') == 'phase3_recurrent_nominal' and active == stage:
            matches.append(run_dir)
    if not matches:
        raise FileNotFoundError(f'No complete {stage} run is restored.')
    return max(matches, key=lambda path: path.name)
print('Selected Phase 2 and completed Phase 3 checkpoints restored; lightweight context restored for all runs.')

# 8. Workspace artifact audit

In [ ]:
!python -u -m scripts.audit_workspace --repository-root .

# 9. Compile updated reports

In [ ]:
!python -u -m scripts.export_phase2_report
if shutil.which('latexmk'):
    !latexmk -pdf -interaction=nonstopmode -halt-on-error -outdir=reports reports/main.tex
    !latexmk -pdf -interaction=nonstopmode -halt-on-error -outdir=reports reports/surrogate_notes.tex
else:
    print('latexmk unavailable; sources and generated tables remain authoritative.')

# 10. Run Ruff and pytest

In [ ]:
!ruff check robosoccer scripts tests
!python -u -m pytest -q

# 11. Run Phase 3 smoke test

In [ ]:
from robosoccer.artifacts import resolve_run_pointer, sync_run_to_drive
!python -u -m scripts.train --config configs/phase3_smoke.yaml --seed 3
assert _exit_code == 0
smoke_run = resolve_run_pointer(RUNS_ROOT / 'latest_phase3_smoke.txt', REPO_DIR)
sync_run_to_drive(smoke_run, DRIVE_PROJECT)
print(smoke_run)

# 12. Calibrate abstract and Pymunk benchmark hardness

In [ ]:
CALIBRATION_DIR = RUNS_ROOT / f'phase3_calibration_seed{DEV_SEED}'
!python -u -m scripts.calibrate_phase3 --config configs/phase3_base.yaml --output-dir "$CALIBRATION_DIR" --episodes "$CALIBRATION_EPISODES" --seed-base 310000
assert _exit_code == 0
merge_artifact_directory(CALIBRATION_DIR, DRIVE_PROJECT / 'runs' / CALIBRATION_DIR.name, include_training_artifacts=True)

# 13. Display benchmark-gate result

In [ ]:
CALIBRATION_SUMMARY = CALIBRATION_DIR / 'calibration_summary.json'
calibration = json.loads(CALIBRATION_SUMMARY.read_text())
display(calibration['checks'])
assert calibration['training_authorized'], 'Gate A failed: do not train or weaken thresholds.'

# 14. Run throughput benchmark

In [ ]:
THROUGHPUT_PATH = RUNS_ROOT / 'phase3_throughput.json'
!python -u -m scripts.benchmark_throughput --config configs/phase3_recurrent_nominal.yaml --num-envs 32 64 128 256 512 --updates 2 --output "$THROUGHPUT_PATH"
merge_artifact_directory(RUNS_ROOT, DRIVE_PROJECT / 'runs', skipped_relative_paths=[], include_training_artifacts=False)

# 15. Select `num_envs` from the benchmark

In [ ]:
throughput = json.loads(THROUGHPUT_PATH.read_text())
best = max(throughput['rows'], key=lambda row: row['effective_transitions_per_second'])
NUM_ENVS = int(best['num_envs'])
print({'selected_num_envs': NUM_ENVS, 'measurement': best})

# 16. Train recurrent nominal Stage A

In [ ]:
!python -u -m scripts.train --config configs/phase3_recurrent_nominal.yaml --seed "$DEV_SEED" --num-envs "$NUM_ENVS" --stage stage_a --calibration-summary "$CALIBRATION_SUMMARY"
assert _exit_code == 0
stage_a_run = phase3_stage_run('stage_a')
sync_run_to_drive(stage_a_run, DRIVE_PROJECT)

# 17. Evaluate Stage A

In [ ]:
stage_a_run = phase3_stage_run('stage_a')
!python -u -m scripts.evaluate_phase3 --run-dir "$stage_a_run" --checkpoint best_nominal --simulator pymunk --scenario phase3_2v2_open --episodes 100 --seed-base 330000 --prefix stage_a_2v2_open
sync_run_to_drive(stage_a_run, DRIVE_PROJECT)

# 18. Train Stage B from Stage A

In [ ]:
stage_a_run = phase3_stage_run('stage_a')
stage_a_checkpoint = stage_a_run / 'models' / 'best_nominal_checkpoint.pt'
!python -u -m scripts.train --config configs/phase3_recurrent_nominal.yaml --seed "$DEV_SEED" --num-envs "$NUM_ENVS" --stage stage_b --resume "$stage_a_checkpoint" --calibration-summary "$CALIBRATION_SUMMARY"
assert _exit_code == 0
stage_b_run = phase3_stage_run('stage_b')
sync_run_to_drive(stage_b_run, DRIVE_PROJECT)

# 19. Evaluate pass-required cooperation

In [ ]:
stage_b_run = phase3_stage_run('stage_b')
!python -u -m scripts.evaluate_phase3 --run-dir "$stage_b_run" --checkpoint best_nominal --simulator pymunk --scenario phase3_2v2_pass_required --episodes 100 --seed-base 331000 --prefix stage_b_pass_required
sync_run_to_drive(stage_b_run, DRIVE_PROJECT)

# 20. Train Stage C from Stage B

In [ ]:
stage_b_run = phase3_stage_run('stage_b')
stage_b_checkpoint = stage_b_run / 'models' / 'best_nominal_checkpoint.pt'
!python -u -m scripts.train --config configs/phase3_recurrent_nominal.yaml --seed "$DEV_SEED" --num-envs "$NUM_ENVS" --stage stage_c --resume "$stage_b_checkpoint" --calibration-summary "$CALIBRATION_SUMMARY"
assert _exit_code == 0
stage_c_run = phase3_stage_run('stage_c')
sync_run_to_drive(stage_c_run, DRIVE_PROJECT)

# 21. Train Stage D from Stage C

In [ ]:
stage_c_run = phase3_stage_run('stage_c')
stage_c_checkpoint = stage_c_run / 'models' / 'best_nominal_checkpoint.pt'
!python -u -m scripts.train --config configs/phase3_recurrent_nominal.yaml --seed "$DEV_SEED" --num-envs "$NUM_ENVS" --stage stage_d --resume "$stage_c_checkpoint" --calibration-summary "$CALIBRATION_SUMMARY"
assert _exit_code == 0
nominal_run = phase3_stage_run('stage_d')
sync_run_to_drive(nominal_run, DRIVE_PROJECT)

# 22. Evaluate full nominal curriculum

In [ ]:
nominal_run = phase3_stage_run('stage_d')
!python -u -m scripts.evaluate_phase3_gates --gate b --run-dir "$nominal_run" --checkpoint best_nominal --episodes 100 --seed-base 350000
assert _exit_code == 0, 'Gate B failed: do not start CC-FDR.'
sync_run_to_drive(nominal_run, DRIVE_PROJECT)

# 23. Record a 2v2 match video

In [ ]:
nominal_run = phase3_stage_run('stage_d')
!python -u -m scripts.record_phase3_video --run-dir "$nominal_run" --checkpoint best_nominal --simulator pymunk --scenario phase3_2v2_pass_required --episodes 3 --seconds 20 --seed-base 340000
sync_run_to_drive(nominal_run, DRIVE_PROJECT)

# 24. Record a 3v2 match video

In [ ]:
nominal_run = phase3_stage_run('stage_d')
!python -u -m scripts.record_phase3_video --run-dir "$nominal_run" --checkpoint best_nominal --simulator pymunk --scenario phase3_3v2_press --episodes 3 --seconds 20 --seed-base 341000
sync_run_to_drive(nominal_run, DRIVE_PROJECT)

# 25. Train CC-FDR from the best nominal checkpoint

In [ ]:
nominal_run = phase3_stage_run('stage_d')
nominal_checkpoint = nominal_run / 'models' / 'best_nominal_checkpoint.pt'
!python -u -m scripts.train --config configs/phase3_cc_fdr.yaml --seed "$DEV_SEED" --num-envs "$NUM_ENVS" --warm-start "$nominal_checkpoint" --calibration-summary "$CALIBRATION_SUMMARY"
assert _exit_code == 0
cc_fdr_run = resolve_run_pointer(RUNS_ROOT / 'latest_phase3_cc_fdr.txt', REPO_DIR)
sync_run_to_drive(cc_fdr_run, DRIVE_PROJECT)

# 26. Evaluate CC-FDR

In [ ]:
cc_fdr_run = resolve_run_pointer(RUNS_ROOT / 'latest_phase3_cc_fdr.txt', REPO_DIR)
!python -u -m scripts.evaluate_phase3 --run-dir "$cc_fdr_run" --checkpoint best_composite --simulator pymunk --scenario phase3_2v2_pass_required --episodes 100 --seed-base 360000 --prefix cc_fdr_pass_required
sync_run_to_drive(cc_fdr_run, DRIVE_PROJECT)

# 27. Compare recurrent nominal with CC-FDR

In [ ]:
nominal_run = phase3_stage_run('stage_d')
cc_fdr_run = resolve_run_pointer(RUNS_ROOT / 'latest_phase3_cc_fdr.txt', REPO_DIR)
!python -u -m scripts.evaluate_phase3_gates --gate c --nominal-run "$nominal_run" --cc-fdr-run "$cc_fdr_run" --episodes 100 --seed-base 370000
gate_c_path = cc_fdr_run / 'eval' / 'phase3_gate_c.json'

# 28. Run Gate C

In [ ]:
gate_c = json.loads(gate_c_path.read_text())
display(gate_c['checks'])
assert gate_c['passed'], 'Gate C failed: final confirmation remains disabled.'

# 29. Record robustness and failure videos

In [ ]:
cc_fdr_run = resolve_run_pointer(RUNS_ROOT / 'latest_phase3_cc_fdr.txt', REPO_DIR)
!python -u -m scripts.record_phase3_video --run-dir "$cc_fdr_run" --checkpoint best_composite --simulator pymunk --scenario phase3_3v2_press --profile combined --episodes 3 --seconds 20 --seed-base 380000
sync_run_to_drive(cc_fdr_run, DRIVE_PROJECT)

# 30. Update and compile reports

In [ ]:
!python -u -m scripts.export_phase2_report
if shutil.which('latexmk'):
    !latexmk -pdf -interaction=nonstopmode -halt-on-error -outdir=reports reports/main.tex
    !latexmk -pdf -interaction=nonstopmode -halt-on-error -outdir=reports reports/surrogate_notes.tex

# 31. Sync all artifacts to Drive

In [ ]:
from robosoccer.artifacts import sync_workspace_to_drive
sync_workspace_to_drive(DRIVE_PROJECT, REPO_DIR)

# 32. Optional final three-seed confirmation section, disabled by default

In [ ]:
if not FINAL_CONFIRMATION_ENABLED:
    print('Disabled. Enable only after saved Gates A, B, and C all pass; seed 3 is never confirmation.')
else:
    gate_a = json.loads(CALIBRATION_SUMMARY.read_text())
    gate_b = json.loads((nominal_run / 'eval' / 'phase3_gate_b.json').read_text())
    gate_c = json.loads((cc_fdr_run / 'eval' / 'phase3_gate_c.json').read_text())
    assert gate_a['training_authorized'] and gate_b['passed'] and gate_c['passed']
    for seed in [4, 5, 6]:
        print(f'SUPERVISED JOB seed {seed}: rerun Stage A--D nominal, then CC-FDR and locked evaluations with --seed {seed}.')
    print('Launch each seed as a separate supervised job; this notebook never starts all three automatically.')

# 33. Optional future-platform feasibility notes

In [ ]:
print('A RoboCup 2D/Webots bridge is future work only after calibrated team play, recurrent nominal competence, CC-FDR non-inferiority, and sustained 3v2 video evidence.')

# 34. Finished note

In [ ]:
print('Phase 3 workflow finished. Report only gates backed by saved JSON artifacts; smoke runs are not scientific results.')